In [ ]:
cat > pv.yaml << 'OF'
apiVersion: v1
kind: PersistentVolume
metadata:
  name: clickhouse-iscsi-pv
spec:
  storageClassName: "manual-block" # Explicit matching class
  capacity:
    storage: 20Gi
  volumeMode: Block
  accessModes:
    - ReadWriteOnce
  persistentVolumeReclaimPolicy: Retain
  local:
    path: /dev/sdb
  nodeAffinity:
    required:
      nodeSelectorTerms:
      - matchExpressions:
        - key: kubernetes.io/hostname
          operator: In
          values:
          - worker-1
          - worker-2
          - worker-3
OF
kubectl apply -f pv.yaml

In [ ]:
cat > clickhouse-deployment.yaml << 'OF'
apiVersion: apps/v1
kind: StatefulSet
metadata:
  name: clickhouse
spec:
  serviceName: "clickhouse"
  replicas: 1
  selector:
    matchLabels:
      app: clickhouse
  template:
    metadata:
      labels:
        app: clickhouse
    spec:
      containers:
      - name: clickhouse
        image: clickhouse/clickhouse-server:latest
        ports:
        - containerPort: 8123
          name: http
        - containerPort: 9000
          name: client
        volumeDevices:
        - name: clickhouse-storage
          devicePath: /dev/clickhouse-disk
  volumeClaimTemplates:
  - metadata:
      name: clickhouse-storage
    spec:
      storageClassName: "manual-block" # Matches the PV exactly
      accessModes: [ "ReadWriteOnce" ]
      volumeMode: Block
      resources:
        requests:
          storage: 20Gi
OF
kubectl apply -f clickhouse-deployment.yaml

In [ ]:
kubectl get pv,pvc,pods -o wide